In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


print("Libraries imported successfully!")

In [ ]:
books = pd.read_csv("../Data/BX-Books.csv",sep=';', on_bad_lines = 'skip', encoding="latin-1", low_memory = False)
books.shape
books.info()
books.head(5)
books.columns
books = books[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher','Image-URL-S']]
books.rename(columns={'Book-Title':'Title', 'Book-Author':'Author', 'Year-Of-Publication':'Year', 'Publisher':'Publisher', 'Image-URL-S':'Image_url'}, inplace=True)

In [ ]:
users = pd.read_csv("../Data/BX-Users.csv", sep=";", on_bad_lines = 'skip', encoding = 'latin-1', low_memory = False)
users.shape
users.head()
users.columns

In [ ]:
ratings = pd.read_csv("../Data/BX-Book-Ratings.csv", sep=";", on_bad_lines = 'skip', encoding = 'latin-1', low_memory = False)
ratings.shape
ratings.head()
ratings.columns

In [ ]:
ratings.rename(columns={
    'User-ID':'user_id',
    'Book-Rating':'rating'
}, inplace=True)

In [ ]:
ratings['user_id'].value_counts()

In [ ]:
ratings['user_id'].unique().shape

In [ ]:
x = ratings['user_id'].value_counts()>200
print(x)

In [ ]:
x[x].shape

In [ ]:
y=x[x].index
y

In [ ]:
ratings = ratings[ratings['user_id'].isin(y)]

In [ ]:
ratings.head()

In [ ]:
ratings.shape

In [ ]:
rating_with_books = ratings.merge(books, on='ISBN')

In [ ]:
rating_with_books.head()

In [ ]:
rating_with_books.shape

In [ ]:
num_rating = rating_with_books.groupby('Title')['rating'].count().reset_index()
num_rating.head()

In [ ]:
num_rating.rename(columns={
    'rating':'num_rating'
},inplace=True)

In [ ]:
num_rating.head()

In [ ]:
rating_with_books.head()

In [ ]:
final_rating = rating_with_books.merge(num_rating, on='Title')
final_rating.head()

In [ ]:
final_rating = final_rating[final_rating['num_rating']>=50]
final_rating.sample()
final_rating.shape

In [ ]:
final_rating.drop_duplicates(['user_id','Title'], inplace=True)
final_rating.shape

In [ ]:
book_pivot = final_rating.pivot_table(index='Title', columns='user_id', values='rating')
book_pivot

In [ ]:
book_pivot.fillna(0, inplace=True)

In [ ]:
from scipy.sparse import csr_matrix

In [ ]:
book_sparse = csr_matrix(book_pivot)
book_sparse

In [ ]:
from sklearn.neighbors import NearestNeighbors
model = NearestNeighbors(algorithm='brute')
model.fit(book_sparse)

In [ ]:
#Using a sample record 
#User ID 237 and all columns belonging to the user
distance, suggestion = model.kneighbors(book_pivot.iloc[237,:].values.reshape(1,-1), n_neighbors = 6)
print(distance)
print(suggestion)

In [ ]:
#Printing the suggested book titles from 'suggested'
for i in range(len(suggestion)):
    print(book_pivot.index[suggestion[i]])
    #Book Title is the index of the book_pivot Table.

In [ ]:
books_name = book_pivot.index

In [ ]:
import pickle
pickle.dump(model, open('../backend/artifacts/model.pkl','wb'))
pickle.dump(books_name, open('../backend/artifacts/books_name.pkl','wb'))
pickle.dump(final_rating, open('../backend/artifacts/final_rating.pkl','wb'))
pickle.dump(book_pivot, open('../backend/artifacts/book_pivot.pkl','wb'))


In [ ]:
def recommend_book(book_name):
    book_list = books_name.tolist()
    if book_name in book_list:
        index = book_list.index(book_name)
        distance, suggestion = model.kneighbors(book_pivot.iloc[index,:].values.reshape(1,-1), n_neighbors = 6)
        print("The suggested books are:")
        for i in range(len(suggestion)):
            books = book_pivot.index[suggestion[i]]
            for j in books:
                print(j)
    else:
        print("Book not found in the database. Please check the spelling or try another book.")

In [ ]:
book_name = "The Great Gatsby"
recommend_book(book_name)

In [ ]:

# Step 1: Calculate Average Rating of each book
average_rating = (
    ratings.groupby("ISBN")["rating"]
    .mean()
    .reset_index()
)

# Rename column
average_rating.rename(
    columns={"rating": "Average_Rating"},
    inplace=True
)

# Step 2: Merge with books dataframe
book_details = books.merge(
    average_rating,
    on="ISBN",
    how="left"
)

# Step 3: Keep only required columns
book_details = book_details[
    [
        "ISBN",
        "Title",
        "Author",
        "Year",
        "Average_Rating",
        "Image_url"
    ]
]

# Step 4: Replace missing ratings with 0
book_details["Average_Rating"] = (
    book_details["Average_Rating"]
    .fillna(0)
)

# Step 5: Round ratings to 2 decimal places
book_details["Average_Rating"] = (
    book_details["Average_Rating"]
    .round(2)
)

# Step 6: Remove duplicate books
book_details = book_details.drop_duplicates(
    subset="ISBN"
)

# Step 7: Reset index
book_details.reset_index(
    drop=True,
    inplace=True
)

# Step 8: Display dataset
print(book_details.head())

print("\nDataset Shape:", book_details.shape)

print("\nColumns:")
print(book_details.columns)

print("\nInformation:")
book_details.info()

In [ ]:
print(book_details.isnull().sum())

In [ ]:
# Clean missing values
book_details["Author"] = book_details["Author"].fillna("Unknown Author")
book_details["Average_Rating"] = book_details["Average_Rating"].fillna(0)

# Clean Year
book_details["Year"] = pd.to_numeric(book_details["Year"], errors="coerce")
book_details["Year"] = book_details["Year"].fillna(0).astype(int)

In [ ]:
book_details.to_csv("Data/Book-Details.csv", index=False)

print("book_details.csv created successfully!")

In [ ]:
book_details.head(50)

In [58]:
print(book_pivot.index.name)
print(book_pivot.columns.name)
print(final_rating.columns)


Title
user_id
Index(['user_id', 'ISBN', 'rating', 'Title', 'Author', 'Year', 'Publisher',
       'Image_url', 'num_rating'],
      dtype='str')


NameError: name 'books_name' is not defined